# Run all debates: pickle-only visual pipeline

This notebook runs `run_all_debates_pickle_only_clean_outputs.py`. It uses only features stored in the `*_visual.pkl` files for identity/movement/emotions. It does not use InsightFace.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
print("Project root:", PROJECT_ROOT)
print("Data files:")
for p in sorted((PROJECT_ROOT / "data").glob("*_visual.pkl"))[:5]:
    print(" -", p.name)
print("...")

## Test one debate first

In [ ]:
!python run_all_debates_pickle_only_clean_outputs.py --project-root . --limit 1 --force-rerun

## Inspect outputs from the test

In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path("outputs/multimodal_exports")
for name in [
    "01_candidate_name_mapping_all_debates.csv",
    "02_movement_10s_all_debates.csv",
    "03_emotions_10s_all_debates.csv",
    "04_topics_10s_all_debates.csv",
]:
    path = OUT / name
    print("\n", name, "exists=", path.exists())
    if path.exists():
        df = pd.read_csv(path)
        print(df.shape)
        display(df.head())

## Run all debates

Run this after the one-debate test looks correct.

In [ ]:
!python run_all_debates_pickle_only_clean_outputs.py --project-root . --force-rerun

In [ ]:
# ============================================================
# LOAD EXISTING PICKLE-ONLY OUTPUTS FOR VENTURA VS SEGURO
# No rerun. Uses the CSV + cached solver PKLs.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont

import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path(".").resolve()
OUT_DIR = PROJECT_ROOT / "outputs" / "multimodal_exports"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"

DEBATE_ID = "Ventura_vs_Seguro_November_17"

mapping_csv = OUT_DIR / "01_candidate_name_mapping_all_debates.csv"

predictions_pkl = PROCESSED_DIR / f"{DEBATE_ID}_identity_solver_PICKLE_ONLY_SINGLE_UNCONSTRAINED_CLEANED_predictions.pkl"
frames_pkl = PROCESSED_DIR / f"{DEBATE_ID}_identity_solver_PICKLE_ONLY_SINGLE_UNCONSTRAINED_CLEANED_frames.pkl"

print("Mapping CSV:", mapping_csv, "exists:", mapping_csv.exists())
print("Predictions PKL:", predictions_pkl, "exists:", predictions_pkl.exists())
print("Frames PKL:", frames_pkl, "exists:", frames_pkl.exists())

if not mapping_csv.exists():
    raise FileNotFoundError("Name mapping CSV not found. Run the pipeline first.")

if not predictions_pkl.exists() or not frames_pkl.exists():
    print("\nCached frame-level PKLs were not found.")
    print("Run this first:")
    print("!python run_all_debates_pickle_only_clean_outputs.py --project-root . --limit 1 --force-rerun")
    raise FileNotFoundError("Missing cached prediction/frame PKLs.")

mapping_df = pd.read_csv(mapping_csv)

display(mapping_df.head())

# Keep only this debate
this_mapping = mapping_df[mapping_df["debate_id"] == DEBATE_ID].copy()

if len(this_mapping) == 0:
    print("Available debate_id values:")
    display(mapping_df["debate_id"].drop_duplicates().sort_values())
    raise ValueError(f"No mapping found for debate_id={DEBATE_ID}")

visual_to_name_map = dict(
    zip(this_mapping["visual_label"], this_mapping["candidate"])
)

print("Visual label -> candidate map:")
print(visual_to_name_map)

# Load cached frame-level outputs
out = pd.read_pickle(predictions_pkl)
df_solver = pd.read_pickle(frames_pkl)

out_named = out.copy()
out_named["display_label"] = (
    out_named["person_label"]
    .map(visual_to_name_map)
    .fillna(out_named["person_label"])
)

print("Loaded out_named:", out_named.shape)
display(out_named[["frame_num", "person_label", "display_label", "confidence", "assignment_source"]].head())

print("Frame table:", df_solver.shape)

In [ ]:
# ============================================================
# FIX: make solver available for the widget
# ============================================================

from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path(".").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import visual_identity_solver

importlib.reload(visual_identity_solver)

solver = visual_identity_solver

print("Loaded solver from:", solver.__file__)

In [ ]:
# ============================================================
# WIDGET: show Ventura vs Seguro using existing CSV name mapping
# ============================================================

def resolve_frame_path_for_widget(frame_value):
    path = solver.resolve_frame_path(
        PROJECT_ROOT,
        Path("Frames"),
        frame_value,
    )

    if path is not None and path.exists():
        return path

    return None


def get_font(size=18):
    try:
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except Exception:
        return ImageFont.load_default()


def draw_text(draw, x, y, text, fill=(255, 255, 0), font_size=18):
    font = get_font(font_size)
    box = draw.textbbox((x, y), text, font=font)

    draw.rectangle(
        [box[0] - 4, box[1] - 4, box[2] + 4, box[3] + 4],
        fill=(0, 0, 0),
    )

    draw.text(
        (x, y),
        text,
        fill=fill,
        font=font,
    )


def draw_box(draw, bbox, outline=(255, 255, 0), width=4):
    try:
        x1, y1, x2, y2 = [float(v) for v in bbox]
        draw.rectangle([x1, y1, x2, y2], outline=outline, width=width)
    except Exception:
        pass


def get_frame_row(frame_value):
    subset = df_solver[df_solver["Frame"] == frame_value]

    if len(subset) == 0:
        return None

    return subset.iloc[0]


def annotate_existing_output_frame(
    frame_value,
    show_body_boxes=True,
    show_face_boxes=True,
    show_identity=True,
):
    path = resolve_frame_path_for_widget(frame_value)

    if path is None:
        img = Image.new("RGB", (1280, 720), color=(30, 30, 30))
        frame_path = None
    else:
        img = Image.open(path).convert("RGB")
        frame_path = path

    draw = ImageDraw.Draw(img)

    frame_row = get_frame_row(frame_value)
    frame_dets = out_named[out_named["frame"] == frame_value].copy()
    frame_dets = frame_dets.sort_values("face_cx")

    # Body boxes
    if show_body_boxes and frame_row is not None:
        poses = frame_row["Poses"] if isinstance(frame_row["Poses"], list) else []

        for i, pose in enumerate(poses):
            if not isinstance(pose, dict) or "bbox" not in pose:
                continue

            bbox = pose["bbox"]
            draw_box(draw, bbox, outline=(0, 150, 255), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]
            draw_text(
                draw,
                x1,
                max(0, y1 - 22),
                f"body_{i}",
                fill=(0, 180, 255),
                font_size=13,
            )

    # Face boxes
    if show_face_boxes and frame_row is not None:
        faces = frame_row["Fer"] if isinstance(frame_row["Fer"], list) else []

        for i, face in enumerate(faces):
            if not isinstance(face, dict) or "bbox" not in face:
                continue

            bbox = face["bbox"]
            draw_box(draw, bbox, outline=(0, 255, 0), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]
            draw_text(
                draw,
                x1,
                y2 + 4,
                f"face_{i}",
                fill=(0, 255, 0),
                font_size=13,
            )

    # Final identity labels with real names
    if show_identity:
        for _, r in frame_dets.iterrows():
            bbox = [r["face_x1"], r["face_y1"], r["face_x2"], r["face_y2"]]

            visual_label = str(r["person_label"])
            display_label = str(r["display_label"])
            conf = float(r["confidence"])

            label_text = f"{display_label} ({visual_label}) {conf:.2f}"

            draw_box(draw, bbox, outline=(255, 255, 0), width=5)

            x1, y1, x2, y2 = bbox
            draw_text(
                draw,
                x1,
                max(0, y1 - 46),
                label_text,
                fill=(255, 255, 0),
                font_size=18,
            )

    # Bottom info
    if len(frame_dets) > 0:
        first = frame_dets.iloc[0]
        info = (
            f"frame={int(first['frame_num'])} | "
            f"faces={int(first['n_faces'])} | "
            f"poses={int(first['n_poses'])} | "
            f"source={first['assignment_source']}"
        )
    elif frame_row is not None:
        info = (
            f"frame={int(frame_row['frame_num'])} | "
            f"faces={int(frame_row['n_faces'])} | "
            f"poses={int(frame_row['n_poses'])} | "
            f"No solver detections"
        )
    else:
        info = "No detections"

    draw_text(
        draw,
        10,
        img.size[1] - 35,
        info,
        fill=(255, 255, 255),
        font_size=14,
    )

    return img, frame_path


frame_table = (
    df_solver[["Frame", "frame_num", "n_faces", "n_poses"]]
    .drop_duplicates()
    .sort_values("frame_num")
    .reset_index(drop=True)
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_table) - 1,
    step=1,
    description="Frame",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)

show_body_checkbox = widgets.Checkbox(
    value=True,
    description="Body boxes",
)

show_face_checkbox = widgets.Checkbox(
    value=True,
    description="Face boxes",
)

show_identity_checkbox = widgets.Checkbox(
    value=True,
    description="Names",
)

jump_frame = widgets.IntText(
    value=0,
    description="Go to sec:",
)

jump_button = widgets.Button(
    description="Jump",
    button_style="success",
)

output = widgets.Output()

mapping_html = widgets.HTML(
    value=(
        "<h3>Ventura vs Seguro — existing CSV name mapping</h3>"
        "<p>Using already-generated mapping CSV. No solver rerun here.</p>"
        + this_mapping[["visual_label", "candidate", "mapping_score", "is_unresolved"]].to_html(index=False)
    )
)


def jump_to_frame(_):
    target = int(jump_frame.value)
    idx = (frame_table["frame_num"] - target).abs().idxmin()
    frame_slider.value = int(idx)


jump_button.on_click(jump_to_frame)


def update_frame(change=None):
    idx = int(frame_slider.value)
    row = frame_table.iloc[idx]

    frame_value = row["Frame"]

    with output:
        clear_output(wait=True)

        img, frame_path = annotate_existing_output_frame(
            frame_value,
            show_body_boxes=show_body_checkbox.value,
            show_face_boxes=show_face_checkbox.value,
            show_identity=show_identity_checkbox.value,
        )

        plt.figure(figsize=(13, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.show()

        frame_dets = out_named[out_named["frame"] == frame_value].sort_values("face_cx")

        print("frame_num:", int(row["frame_num"]))
        print("frame:", frame_value)
        print("resolved image path:", frame_path)
        print("cleaned faces:", int(row["n_faces"]))
        print("cleaned bodies:", int(row["n_poses"]))
        print("solver detections:", len(frame_dets))

        display_cols = [
            "face_idx_lr",
            "person_label",
            "display_label",
            "confidence",
            "assignment_source",
            "model_person",
            "model_confidence",
            "unconstrained_model_person",
            "unconstrained_model_confidence",
            "weak_source",
        ]

        display_cols = [c for c in display_cols if c in frame_dets.columns]

        if len(frame_dets) > 0:
            display(frame_dets[display_cols])
        else:
            print("No detections in this frame.")


frame_slider.observe(update_frame, names="value")
show_body_checkbox.observe(update_frame, names="value")
show_face_checkbox.observe(update_frame, names="value")
show_identity_checkbox.observe(update_frame, names="value")

display(
    widgets.VBox(
        [
            mapping_html,
            frame_slider,
            widgets.HBox(
                [
                    show_body_checkbox,
                    show_face_checkbox,
                    show_identity_checkbox,
                ]
            ),
            widgets.HBox([jump_frame, jump_button]),
            output,
        ]
    )
)

update_frame()